# Assignment 3: Fine-tune GPT-2 for Creative Story Generation

This notebook demonstrates how to fine-tune GPT-2 on a story dataset to generate creative narratives.

In [3]:
# Cell 1: Install Dependencies
!pip install transformers datasets torch accelerate

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Cell 2: Import Libraries
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch

print("✓ Libraries imported successfully")

C:\Users\shaik\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries imported successfully


In [5]:
# Cell 3: Load Pre-trained Model
model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Loaded {model_name} model")
print(f"Model parameters: {model.num_parameters():,}")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 477.38it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Loaded gpt2 model
Model parameters: 124,439,808


In [6]:
# Cell 4: Load Story Dataset
dataset = load_dataset("roneneldan/TinyStories", split="train[:5000]")  # Small subset for demo

print(f"✓ Loaded {len(dataset)} stories")
print(f"\nSample story:\n{dataset[0]['text'][:200]}...")

✓ Loaded 5000 stories

Sample story:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on...


In [7]:
# Cell 5: Tokenize Dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print("✓ Dataset tokenized")
print(f"Sample tokens: {tokenized_datasets[0]['input_ids'][:20]}")

✓ Dataset tokenized
Sample tokens: [3198, 1110, 11, 257, 1310, 2576, 3706, 20037, 1043, 257, 17598, 287, 607, 2119, 13, 1375, 2993, 340, 373, 2408]


In [ ]:
# Cell 6: Configure Training
training_args = TrainingArguments(
    output_dir="./story-gpt2",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print("✓ Training configuration ready")

✓ Training configuration ready


: 

In [ ]:
# Cell 7: Train Model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()
print("✓ Training complete!")

Starting training...


C:\Users\shaik\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


In [ ]:
# Cell 8: Generate Stories
def generate_story(prompt, max_length=200):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        num_return_sequences=1,
        temperature=0.8,
        do_sample=True,
        top_k=50,
        top_p=0.95,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("✓ Story generation function ready")

✓ Story generation function ready


In [ ]:
# Cell 9: Test Story Generation
prompts = [
    "Once upon a time in a magical forest,",
    "The brave knight embarked on a quest to",
    "In a futuristic city, a robot discovered",
]

for prompt in prompts:
    print(f"\n{'='*60}")
    print(f"Prompt: {prompt}")
    print(f"{'='*60}")
    print(generate_story(prompt))
    
print("\n✓ Story generation complete!")


Prompt: Once upon a time in a magical forest,


NameError: name 'tokenizer' is not defined